<a href="https://colab.research.google.com/github/SamC-MIST4600/OG-Game-Justice/blob/main/OG_Game_Justice.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# OG Game Justice

My app helps Game Studio decide which old games deserve continued developer support/ a remastered edition by sorting to games only order than 10 years than comparing ratings, playtime, revenue, and owners.

In [7]:
from google.colab import drive
drive.mount("/content/drive")

import glob
import os
import time
import pandas as pd

# Look for steam_games.csv in My Drive and in folders up to 3 levels down
found = []
for pattern in ["", "*/", "*/*/", "*/*/*/"]:
    found = found + glob.glob("/content/drive/MyDrive/" + pattern + "steam_games.csv")

if len(found) == 0:
    print("❌ Can't find steam_games.csv in your Drive. Upload it next to this notebook and run this cell again.")
else:
    FOLDER = os.path.dirname(found[0]) + "/"
    games = pd.read_csv(FOLDER + "steam_games.csv")
    print(f"✅ Loaded {len(games)} games from:", FOLDER)

Mounted at /content/drive
✅ Loaded 1000 games from: /content/drive/MyDrive/


# First Subquestion
Release_year
How many games are more than 10 years old?

In [ ]:
def games_over_10_years_old(games):
    old_games = games[games["release_year"] < 2016]
    return len(old_games)

print(games_over_10_years_old(games))

214


#Second Subquestion

Of the games that are older than 10 years what are the top twenty by ratings?


In [ ]:
def top_10_old_games(games):
    old_games = games[games["release_year"] < 2016]
    best = old_games.sort_values("pct_pos_total", ascending=False)
    return best[["name", "release_year", "pct_pos_total"]].head(20)

print(top_10_old_games(games))

                                 name  release_year  pct_pos_total
6                            Terraria          2011             97
17             Euro Truck Simulator 2          2012             97
45      The Binding of Isaac: Rebirth          2014             97
48                       BeamNG.drive          2015             97
203                     Hotline Miami          2012             97
125            Mount & Blade: Warband          2010             97
278                    Papers, Please          2013             97
301                        Dishonored          2012             97
352              BattleBlock Theater®          2014             97
350  South Park™: The Stick of Truth™          2014             97
605                 The Wolf Among Us          2013             97
632                          The Room          2014             97
395                  The Walking Dead          2012             97
893                     Super Hexagon          2012           

# Thrid Subquestion

Of games that are older than 10 years, what are the top twenty by playtime?

In [ ]:
def top_20_old_games_playtime(games):
    old_games = games[games["release_year"] < 2016]
    best = old_games.sort_values("avg_playtime_hours", ascending=False)
    return best[["name", "release_year", "avg_playtime_hours"]].head(20)

print(top_20_old_games_playtime(games))

                                     name  release_year  avg_playtime_hours
254              FINAL FANTASY XIV Online          2014               837.6
2                                  Dota 2          2013               717.2
0                        Counter-Strike 2          2012               553.2
3               Grand Theft Auto V Legacy          2015               322.0
4         Tom Clancy's Rainbow Six® Siege          2015               236.7
280                   Company of Heroes 2          2013               215.8
74                                 Arma 3          2013               188.3
377  Total War: ROME II - Emperor Edition          2013               187.3
8                             Garry's Mod          2006               183.1
64                          Path of Exile          2013               178.0
45          The Binding of Isaac: Rebirth          2014               165.8
937                              Toribash          2014               137.9
23          

#Fourth Subquestion
Of the games that are older than 10 years and have the top twenty play_time, what is the average revenue of the these games.

In [ ]:
def top_20_game_revenue(games):
    old_games = games[games["release_year"] < 2016]
    best = old_games.sort_values("avg_playtime_hours", ascending=False).head(20)

    best["revenue_est"] = best["price"] * best["owners_low"]

    return best[["name", "avg_playtime_hours", "revenue_est"]]

print(top_20_game_revenue(games))

                                     name  avg_playtime_hours  revenue_est
254              FINAL FANTASY XIV Online               837.6   39980000.0
2                                  Dota 2               717.2          0.0
0                        Counter-Strike 2               553.2          0.0
3               Grand Theft Auto V Legacy               322.0          0.0
4         Tom Clancy's Rainbow Six® Siege               236.7   79800000.0
280                   Company of Heroes 2               215.8   24950000.0
74                                 Arma 3               188.3  299900000.0
377  Total War: ROME II - Emperor Edition               187.3  119980000.0
8                             Garry's Mod               183.1  119800000.0
64                          Path of Exile               178.0          0.0
45          The Binding of Isaac: Rebirth               165.8  149900000.0
937                              Toribash               137.9          0.0
23                       

# Web API

In [ ]:
prompt = f"""
Analyze whether {game_name} deserves a remaster.

Release year: {release_year}
Positive reviews: {positive_reviews}%
Average playtime: {playtime} hours
Estimated owners: {owners}
Genre: {genre}

Write exactly 3 sentences explaining
why this game might be a good remaster candidate.
"""

print(response.text)

IndentationError: unexpected indent (2358021012.py, line 35)

# Class __init__
I will use a class __init__ to gave any old game a score out of 10 whether it should be remastered or continue to get updates based on the 3 subquestions

In [22]:
class GameScoring:
    def __init__(self, games, game_name):

        old_games = games[games["release_year"] < 2016].copy()

        # Find the selected game
        game = old_games[old_games["name"] == game_name]

        # Calculate revenue estimates
        old_games["revenue_est"] = (
            old_games["price"] * old_games["owners_low"]
        )

        # Calculate rankings
        rating_rank = old_games["pct_pos_total"].rank(
            ascending=False, method="min"
        )

        playtime_rank = old_games["avg_playtime_hours"].rank(
            ascending=False, method="min"
        )

        revenue_rank = old_games["revenue_est"].rank(
            ascending=False, method="min"
        )

        # Get the selected game's rankings
        index = game.index[0]

        total_games = len(old_games)

        rating_score = (total_games - rating_rank[index] + 1) / total_games * 10
        playtime_score = (total_games - playtime_rank[index] + 1) / total_games * 10
        revenue_score = (total_games - revenue_rank[index] + 1) / total_games * 10

        # Calculate weighted total
        self.total_score = (
            rating_score * 0.30
            + playtime_score * 0.35
            + revenue_score * 0.35
        )


game = GameScoring(games, "Arma 3")
print(round(game.total_score,2))

8.4
